# M8 vehicle detector, step 17b: one training run on the full UVH-26 training split (GPU)

Three runs, fixed in `baselines/2026-10-28_m8_full_train/TARGETS_LOCKED.md`: `s_iisc` (IISc's model, BatchNorm repaired), `s_coco` (stock YOLO11-S), `m_coco` (stock YOLO11-M).
Set **RUN** below and run one per session, in that order. Each trains for a fixed 8 hours, keeps the best hold-out epoch in full precision, and exports ONNX.

**Settings:** Accelerator **GPU** (T4 x2 or P100; one GPU is used), **Internet on**. Inputs: the `uvh26-train-1280` dataset from step 17a (it is unpacked to the local disk first, a few minutes), and the `uvh26_train_kaggle/bundle`
dataset (IISc's weights; the script is inside this notebook). Use **Save Version -> Save & Run All** so the result is kept (a run is limited to 12 h; this one needs about 9).
Output: `uvh26_<RUN>_results.zip` in the version's Output tab. Send it back.

In [ ]:
RUN = "s_iisc"      # s_iisc, then s_coco, then m_coco
TIME_H = 8.0         # fixed in the targets file; do not change

In [ ]:
import glob, os, subprocess, sys, time, json, shutil, base64, zlib, hashlib
assert sys.version_info >= (3, 10), sys.version
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
# The script travels inside this notebook, so a stale copy in a Kaggle dataset cannot be used by mistake.
EMBED = (
    "eNq1fXt320ay5//6FFjckw3goSCSljQOM5xzFEeJfcexvX4kdy7DgwFJkEQEAhgAlERreT/71q+qG2iAoOxMZnVsiUC/q+td1U3b"
    "tj/kQZRY5Tq0fnpm3YbraB6H1iIsw3mZ5lYqRT98fPXK+vjzi9PhpVWiQZSsrCKLo9JyfnrmeicnH4J8FZaFFSQLKwuKwpoFeTGi"
    "30UYR0lYnA37w8vTQf90+MzfPPOX2zj2uaezD1fvfrz+8N5/9eb5366/9zYLy1lG9+HCmoXLNA+px101Jkay6CfLw4z+LMNyvrbC"
    "2zDfWXEwC+OYmsk0T3/6+VSWtsyDTWg5wSzdltZw0Ov3+27PKkr0TfMPSmuTFqU16A2f9a3s3rqLFjRoYf3n2+sfe9ZdHpWh9fc3"
    "r97ICAWPr37uonJtBXFsDc4xrDWPaeVhQb2H1HGBji76fWudxotTjM5TKSynCGmew0uXYEvDp0m8I4jPUR3AnoU0nTBL5+seg3O+"
    "Duc3UjVJzeFvgzhaBGVEm7SMaNcSLDSiPdiktD3oqtorFBWeZb0Li+0mmMUhfX5Nsyi4WpSUYZ6E5bc0gPXj248ej8KNLZpeaOXb"
    "xEqXVDcPCZT/KPwoKub/6Fn0aZ7OU3zayCfXok2zAku2sIxoRrPtglCjV6+tggcv0roJs9LCRhFOmMujTZ5HBa2uZ4X3WZqX6DC1"
    "3rx+/V+Ws9jRkqJ5z7o8x665NaSoVrCimdNAGPHt7kOa0yibdBHG1aoDWaYg065cEwiLeR5lZXFG2MkrZxz1sp1CtlOe8dlNsFrF"
    "4dldmt8QXM8WxRd1IaA8PaWRdQ9Rkm3Ls+3tmnCVi08HhIDUIVUDuAXG9MAwfHZkfKrpq5q//moC7/QUL0/vwmi1LrtHxeQIhU5n"
    "22RBRQyg4qwmHyD97WBw+t7LyhPbtk9Olnm6sXxaVbnNQ9+3og22hUCfpCXjYXFyot/lq4w4QKif10GxjqOZfvytSBP9OS30p5w2"
    "Md1UT1XjYr0to7h62lUNgJDBggBRvSBg6c+fogxkIbOep8Qc5jxHPe3n6RZ4r8uT+TbPw6T0ZHlVtQ88xts0ja/vw/mW+IZuQRur"
    "6izCMMOzlGRBibXqwrf0WMEl2W6oGbGXJDs5eXf99o01tmzeqiDaKODbJx/eXb187b+6+u761XtUUJvCDO2sxeA8wNI++fnq1WGD"
    "n4PYqE5PqvLHVzTCq79/ePn8vf/z9bv3L9+8Rqtn3rk3uHxqn1y9+u7jT9evP1x9oJJGlaHX957ZJy9fvn/uv39xNby4xNvBcDEL"
    "Z8+Wl8tg2P/zYvDN+Tl9vLwYLufnTwdPn158s5yF8+Hln/vDP8+fzfuDy/NBv3++mC8Hi/7gom+f/HT1X/77l99f96zX/os3r75/"
    "8/FDz3p/ff19z3r504/v/5tGAX30wFB7xDuZ8E9+eDu49KkplV5eXPTPvf7Jd6/9764+PH9xDThcnp+c/IfiRUSAB6Lm5N3H16j3"
    "wKRjCyXZI2sRzUunKIO8HPPm+LOEOrGJH2+y8Q9BXIQ9K877475H0uSiR2KOBNF4QLMiWRJEcyqgt/M4LUJfvblwe3oQMMrWILs0"
    "TgeDgihNDfIh3xpjDPQQT4fVEAOv3xriqR5ic3yIzWeHMFbRPcT+BEA9/cM/zFhPTvw4nQcxbUJFzB6/cVwaZxEuLZ9EKsSA4454"
    "eRV7+OeWpElhWf9BYuufwch6++p5/3xwIVw9WtJb5jtBWeaOjNIj8NuqG/zIW6+g0XV33ns9WlWrTG+oQlp4YXIb5WnikThz7Bc/"
    "+B/e/O36tV1XpDGp7sjkw9UQ3poWF+bFxL7akrDIo0/MMe0pdb20vwuDPMytB2q+t7l9HhIXSqrmChas8fizXRkWDtjMiBSZnPSN"
    "6FM4ghi3/q/1GgJ7zH9I5uZRWEgJEVDftU7/anFrVfH3QJTEQ7Cjbi74CWLeB1ER016FDg9kQLbMd0045NSy3kkG4dJel2VWjM7O"
    "1tvVijZ+GcxDb56ekVJDeiMJ0gdwyP0ZceM0viURBfb3gHXvCYfB60kmjgfP+m5jKNqG3COsL7egNdKqxmNrSHoYVAQHoIKSxGCi"
    "NcRh4uQ0KImCpHRRFTXc5uSNDanqfnbI8/750V4w+uc7GH5z2AFW7RUxCRxnEyUO7SxNX+GWIOa7kIB/erUk2UZQ4l1zSeN9Snpv"
    "Dafwfg61q8L6d/Lhml/TFjUHhkZ/0jED6bwq0RiCicnnJ9ZQRhacDqIitN6R2KUurvM8zQkH5uk2XjCtLtK7JE6DhSVbbAVYAtEE"
    "UGsvqGxrplCsA5I9PgS8kzEVEEZD0jKK06MsYE2zUaqHJ00UXbPqnma0+RkxhXxmuxDKy3rZwO/5epvcAMfJBsidONjMFkQUSw98"
    "yhlYf/kLoRVBdmbbLXxZe9uMkDh0uAfXJOg1bdb9IloRsGsOF6VbJ6COePJLgkGpCPMe0AzunT44McAaTIZTqki/qS4XBZM+v+lP"
    "1fZGu8NGT7nKU6PRgN8MqkZQhagdjfiEeuB32wSmxdjiQakhRnKpmPvjZ2pu/clyZlI+0+UzKefu6S/3bcJABjtTAxDqy4e/Wn2L"
    "VNDQookryEBo+WxAOqRlEieLo6KcQKzR9H9hxtazXqi/87jw06VIvQm/oV9Uj40f1ZQwY6o2i1Ta52+ev7Fm6T3xQzJvwBGiTbAK"
    "sQu1yWfx+LRqMfSUkcdmC20+I9IdjLM7QqBFnmZZuPCgLmMMbgprsydmztia0ISU5ulQ88m04qYBMI1XaQqUQIialJA8vVsQQfdb"
    "uEaLsrkMC9ETILHyJ2L4jYrgW1GyrfnOfc/a0QJ6TCUOo51zKwbcLc9lYs+oT3taU/g9qQk7+n9PZL0bmnh2T4upHnauIJ70+Qs9"
    "3ROe3DXevqCnHb1dN+Tn/ZAQ5p5oyxqCO+/wuOPHw1VjveEmK3eQBckO9UkeFNqO5q38IkBAvAvyTGjNcyLcVZrv/IjAOG3sRbJz"
    "bsCZb4YsTJhunSZMwA+GLnDZ++aCYXkzxCsAlFHmcB2LbRZHGPV37yDNqT289b9oL2VnGeYayN3gm8cRBoNdfRRkmLUXULVk4Tg3"
    "vYMRDanC6K7rLu2Hm731QNVpAvc0szPa0zOiWu9yifc7vN9V71/o9woF3EZdwQO3qmdoXbSWpc1EPLIemNYnN9O9uQrFdipiVMxl"
    "vln40EGdgPnugT40vx0eUYVgio9Z3DiBRw9uRcRsZxC5ApDFGRv2RLP6mcxtPIknqS5Vzyg19slhg99auN7mZhHlpO/BPC2U3h7e"
    "Ezvz0xt+dE8qbsH6oi/j1X3BHqAZw/rzIGILR01eqssEXBZsfhnek2iq4QsZCw35IfMA3RHpUKRQZ4TogJjwiwzLbnQpM3A9iE7M"
    "3t2fGH4rn/cJnUabiY0R+A1tGnqLNuju2GRvj0x4oqBsT/cnhkqytMBAoVH0ulQ7hRs8AiqpPkXFFiBAJj0GSlMpNy13A4aNJZMl"
    "gZE8OEfx0mnBwO0GgjlMbe03113tGfwQhlbtr7eVP+LF8iqLjiB2Y7/3DWVoielwW4eslDAZHzeIXA+y1ifSSv0yD0PnQIuFXt+z"
    "pMIuC8e2UvmJGgAaP0q49bjp+UAlG83m27yIbkPB/VFbkV4KeOP0DjKWJrkoIKUdx/ayZAWC837L9N9wRfM9VLPjJUBA62LrcQki"
    "XRa2QvmD2mxUjNFIQIFn0ihpLlEBT2SQzEOHSnusm7ii5ejO+b00OdY/b8nkAG9koe6UxbdXozgkoTl1s++6c6oB27PGKi4hOYSV"
    "F+xvdYDslUSE/j2FIrMbiy5szUfWnNBvoVUEjeITemvQ8xyog45FmoqoBYapxqS/Ccr3pGqYbDdhzvozNVKcQ7FDW1zZ7MdyPXbP"
    "CwNgSllsN1nhcBW1UlEba5UQmDmtsZsVr4Tnx0s1fJk0rwpW6MUj/CR2EmzjkpRRRXWsJECNc7XYCxLFjZkomyMJTctImmjrQRLx"
    "MjzOG+opkRYKhyKbTrBRuL+etbTLu7QOh6hpkPGTi1rEUYKHRLkZ6plOEoAl2ohDO03jGgto4Y7i59b/RnDDYYRUEI7D4KZZF61R"
    "s+J6lXxC3XrBYg++3xVluLm+j+AReIA9jlruvo5eVNGNwsIquuIetPDQW3nWA9pORk+n+28R48kyaq90hSyHsayGUMvZ14CqR8ul"
    "P64nC90ru4CfVAkvcm9tC4RTvrVsg2iXdl8BPBBws5EA4HPLGiz7xlp4XKLTZbwt1oZcT/MF22YYcTKaKlMarnLvHf9x4Ch1ycDd"
    "LpdkD3N9RZA+gi3UNvDwgUhIzGL5jF2j7eL6k5HUVRhW3LBiqCiHnUwtyoGzgihL2TL0vrJqtLuILSpHdAbqweCyEjocWzZUHnbV"
    "cegqqSbGDNIWBclQ9MKNQSLcCSgfjV2yPauKxN3JJLjHCMI0lloBe+BGe/pLfe2VGNB1lBrWqkP92KbCTW08Vr5IPWArgCpULzp1"
    "C8yvJ2vuKb9cw/8DsSbCxnkinB4tpg27iOsot9XnB2GOQbZWUUQs8BrDRZsV7J3boRdtFiGcTU6SedAYZoQ/tIEYi6zmzNsSwTwj"
    "PQ91X/707vrqe/85mcXvGjNDd18+sUVKRAyOBd9Le2abkAGh+RGAUJW9IHufy1Zw5GThZDRs2GbOLz3rhVg/6GZi30WLcg3eLI9r"
    "joTZ0y/YIpaXhPgARiGqFGhXnAGAbHvelWFBE2x5LFgfkBlUkoJW0sN6RA4q/0QDpjCksWDYkToy0jLgaChdQshbNWjqQtVO5yHW"
    "RFyPCMPJiVAXzi/WE6ugucjTC35ye+KbydKYedKYd/71h+t3/hVt/4HwESRiIcxGAdEGHEK2V24yRVw85EQw6Jd3Lz9c+wis+//n"
    "49Wrlx/+3rO+uTAQnUib1L6YFNZj3eFV7YkkyjM0APvXhDS6NCIBAvizawrvWPCwD0fYim0oQcdJFGYiD9InrU186uz5xC94bvpS"
    "mi5gDSgWSiZKtImIS4ophs8yJpfXjsfDmCJZN4jpEsKxIzK8b3oio9rTTtOBEMHALqJgLYRG7cZq7tY7Rk3oVffeJsgciQxQ+0k0"
    "ighIl+fTDg2Y13zgglAISp0etjDkhqkr8QQMBYnZdVenRXmkT0ga7VEtymZbLdUt6wEzJrZdQWfPayBxXWw3DhZOJb1qhiR6tjDx"
    "XKqn3lFVY4utU9r7kddfUrnd65hZW06Xea0GZR5Lq8o21tprwzHgeqs4nTn2E0ZuhZXr9It7YYdBVx+zna+E5cOhtISoHJHeyJiC"
    "XqFXiEyfQ4Szzlpko1qgc9VCZmBXXgsevdZtpVx6aCKvOXVTyu6NyUPKdmAyyJanaFr6shbhsR2IK1OYFNl0It4gIAhqSzsH63en"
    "hntIhQo1wUKmq+dOX4rifUCzdUoceqy1rFNr2If+zQk9D6p8307/EWViE0Dbf7CJTkqS0YU9oocq9E1P0ieBGbodPUss3NY8n97U"
    "IXNb+aTp5eB830BVBW4OlFCx6Z7oKVsKg7HG22gn2zyyJgqXJoXYcgW2o8xJktlKWTtWaZ1Om10qKqP6Fb3Bb+czfRvR6hJiSTx6"
    "ZJj6vDjABzjJdeYaI5W9KDsOV9OGMKIFgFrDFqXbV1lC1GNTGYfgh9K+8MGj/bod1ew3bVDlrfBpF6NlWJSPm6NUC3J1ESbleOA2"
    "jRFwqJEgC2nIyuYhnnwUfb7VwLMenIcbtQaBx01P/Paaw2l4uPsG+zLZ1r8rkh/Mbziegw/U5d8428i6CcOsqDLskAonviUy/gOQ"
    "WDhL05uvCyjg2ZZEVSFeaEnS09lyBJFb0rsCJG0twztSw3KyxMogl848w5dLgx/z5VJ91omPhLaLXtOnu2DfIkedIesNT2/9WnuC"
    "v9Q9KyRvOPQcpIMdR6YDZ6x2zGnh8AhbPvRF135m23DZyhQW9L/IDF9tlxgmLpl5kXASZfmAS4pTuHay1cqadg+AqfbYHgaY51uV"
    "r0AqyJINUy7xNzPSQAf94bn60+NIGf73lbZdWZWckWJy/oQ4LieRlEtjgEZaxrIpJ8qlp3ppatU8Sa2oPIgPa0QsXebIRCl13FH/"
    "fEHGYYAYO5tbhWIoND6RXDNu01rzwaqq/eCORs2Zd1tX3JHCao8D2VrAfmaud0bohMAQLBYIgQf5HIsds2ufUA/6/y2xspQwxIzw"
    "0FI0eGTnjTgMrCYSqhnnMUBMl357H7gOSUxGiuaCzP0wPye+mDHM/tk7RJSlaMe06/cHqtDn9RbtQ8MQE1VjigAfxtnAyafeKXyX"
    "euigWUtLQth2DtfqHXahA+3LVCQ/7w2EoaKMBhaJ36mKHLFCQN02xZBC10eFD8Zz6/hUIsDo5jm9pnfV8NdwAqaHFEcBfKK9JklT"
    "nIEFQzSZrjNhW86DQO9rBsfXU0PUqQICK1635J0LizS1THTe19y/7Xczq1HXT/dKtqhJIKgdxy0XWy1ARH79ayKkyOctGUJvvlCI"
    "KJwwRQM1PtzhA4nwO6SP7JDeeMKFmCzSEYGCJ26f/hUftdXB7riKKVGNWjS4k9HgfHropMzTlC2HCfeHJjIlcdIQ7Fqu8GLNTmta"
    "+USTwrRpNuRs86LbVnINuw4nWT3B3MuZ8ou1Dju4LXE1bceI1lGHsYlZTupO4OykeqZnUf/MaCtuaqc2KW8BLHfFE5qrclumfE+t"
    "vY5yHK1fOQwajL49TfYYtCWcYu+ELHkwLwnrnSpOfWg6C+KDcOEPGCgjmqf1tUzr6ym0yINdb4ZGK8VvE+ysNaltdcd8GkFTrZC0"
    "oGBh3a1DOWWicROJNJtgQXrxikgmzLlQaYPaG7c5aa+itrp4L/bQMjuXYcxjCd/Xt1ZMaih7/ml7qhkfdcezhzmdMbJPDw1doH6u"
    "BJF9GJxs61CZohNPu8APVLlGrYPKh9pdhyuncoR39nPWnMRh8y/lMeYPIFRljZCCseBhWGNwTzqcN9gptKkEBmEI5M2RDYAq3T9p"
    "0EiHU21wqb1pGkE3OE5UbJDLhdqGe1dj3wbG5EhlIxfEAFQ6EWlEYTLfHbgoDl1zvIweUk77XZFlqC/KE6fCqL+NTAnLyPHkN+oB"
    "HSkfHTqbuodgTlS6pa5jzOCwcg1uGicCuScK2kegDM8ImFr/0DNzjDs3aUEESm4QBMmPy/6029GnZBSs9p6VdU+pM25oJ2FUcYqa"
    "vJNDerbuwjwUureiBtv5VnaZdgesIJilxL6KdXpXSHZfJOeWCvsQe9vAFJyqBxY+1wHif49CJqpCWyM7tDY/47s4UC6OKN6HEbU/"
    "qnkDw0QE6pi4qW3/e7Ty+nCUYzdRQjs9Jd5T8wMNH5q40uibc2wqvrUE/dfU3AOd9N/il6li2uswzggHRdWdSXJNXjp85ou1XfjV"
    "mtoun5rr1nVnCQs/cUxzSoOcrqPf2xhO2Vbmy6Yn3XlJ4n2HQyav03wzXCitLL+FJcvlc7IbJxsv3yaYtX8b5N4iLIP5mqxJyRrF"
    "36AkTkyf5tlW+aZ5DjQrtSn55liPmzBI/qUuZwGE6P840mtULAmupLnlt8hzaL/cuI2U7webIB4HOzABoSTqF7bdfB0kCWQ2JzBT"
    "Zx5UwthBHM5O0kT681vVaCoeIhqu2wxKVN35xLpyf5kNLonI71Ur6n3yP9R0av3V0sem3M/0E5THeyFC+0w31Mo3dhJ+FIa37sJD"
    "0NJlXHHwwvWQX6uSo2B94aDR1ccf/b9d/x2HtBxbDiGBk2yi+22GDyTnN3wyC1x452cByQQ8LcJVHopeRESQFJDeeCjmQSwf1qH4"
    "Q0AXGY4G3oatKI+9jKNsu2AHD32Kufq6uPXX+kOhP9ziw2yVV+cTmMZorGjGGj7TR49Fjb8LNnHlDuJjokG+KsBl+ABWWIzrM2zm"
    "oSwVHxwPcb44XIw56aNJu7ZtvwtPiWkRkypDdTK6ojiLcfDrggNqkHPzQp+7rQ/bWs58u9mK+8ciFT5H3gsQyvpHNcF/1OeDwVzU"
    "WyvYrjakIOIArpwr5lWzOs91Q9pi6w3UdBQakwDCrcIqaf5xDlSlN25j6jXeoQdvvlzpZiuSbXj8soaSVyEtZ9soXvh4A4kZ5j31"
    "hkP6Wj388m49iOjqGClHHXxiPb+zJ+7EY1j4jQ7BG/yCTzMfOQFXVeAD5z0+1Z+TsshQN9QQ2jeI+ZuRgY6TG4nk8PGXigiJVG8k"
    "/qPrqbjhciUJLIC8A2zJSW0uxg9PnlDn7MVaFZ/g62KstwnZEDZSJyntObHj0HhewmSVwAuOIHbqikZYKZSYEq/RbiwSES9apnLH"
    "qjybg51wKrJU9SDdDrfeobUJCVcKiCJPedtjfB9X6k5RAgjjp0PVa1z1WqOYA8ef6kPRN9wDnMel7CziXjfj04HqJISwTKClicgl"
    "m4zUB1oyItkk2W6juRhwUqwk3P8HqW3KyFov3cCzRZDVfJ+jeYaTfUOjgUdsN+qIojFXhpuqS5tZW3iNs4TiDpF5pf4qDxbwrcmL"
    "gIy+OUkAh+DgIa+YrNQE6XkLObTbEWCOCPxNZ8wi7jDaABmJ9Cpm123A1G4h84eX58ygOq7sqQefenhbaR84/nBx0ZFTDBD8iVBG"
    "t5O0J+WMItWAbFgDLgLD8JYPzporPNykg43oe/2nps2nD80uYo4tp0XY9ios4kb8BDckjA/0SlPR59XggBc0dqo16dJvWKvoc154"
    "U3+qnODo5pCdKMFDL5jbPHlCHez1QUGSYaTFPB1qGSxp0qbgffKEvWTNk4NfoAbPb9iRz7dY0NinA+gAJHz9ZVQmIcesJZXI5qER"
    "4Vb3BCgAVQooq51UL9wEupEBbjvNSKDTUvO6x3r+OnBev0FPyJWhEjmrWeZLTmmxv/r76Veb068WH756Mfrqp9FX7223Wv5e5TOB"
    "kAA0Z34jwOIMLOIzpTJ9jSStZrFUb6i+Gd99wBsRh6Rj54j3g/tLKprsX80ov/D4DZhlZy5grrwhHGY6s3Q6/Jl1p0xaepNwU6jb"
    "nPi2tp5YuUp8U6/u5JXsMbWYo0WChUrHp/TgWl+peXOPRtFdXaQS6pPbAJwXSZakajmO9NmzntKog8G5kWqptiCDF2cJLzfVVcOd"
    "neEorUNzkTHwbPQ/oVYj+k8zwSLRfMR90PPdtCsLMKEZUMfHM/5evXx9ffWusZ0yltpQkvLRcuenSXLv4JevKItVFUVsOimdD950"
    "2pmdu2yUo+dcTgxDrcSrx6o/Rq8FJ+qV3suEDP2QRJw+6m/MPsvT2wjumvHEfv72o3gSqc5b9b466cCHJyp6NhbdomqTJd/NCHFZ"
    "05dAEZ+V5F+VKKxcZ+2sI5wEbpGQZF/yYWScxmIYN+oTXgUFHxpcbdNtEeR5sHPuJ6OehX+j08HUY+MoAycfckx64E4wF9JsFnww"
    "B0iL5TwdKkHl1V5XKFMFTGvntcqzQ7arz3fMFJxmpQ6M3e8b6dldIrwpXmas5JTK2oZO7PPFKc694dVEJQdnjlt6CwGZXVWEitss"
    "Dl1lUc5cT/o4yGINhItA+MzUR/YrZbVJH5BxSqSL8K6q3dN1DZ+02lZVYzIwljxTZ2WBAoIf2JxZ4QS8HedyXlo+usoyrnu+K3Tz"
    "oqv5+ahqTh+bzdUaaQJ/Aapd8NKov79Yg/D0Ka3Hri5XEnNtERUB2871qYU0j1bQN4jDCwq33BvKhSW+DXWygmSQhF4ldp0iv4dd"
    "AkBf+Nr8DE4FxXGp1wtXVSCMzUOuUrkMlvbDXTHyhuEe/kNhP5vgJvSV5JcDO059sPtnomAYs0+/LtTRDTgEAsnnKixnEW1IThAb"
    "TNIIJscsRooGrlGIkSZureIgD91v1SVg6+CToAS78gjNdhZxJDFp21ZrEM+29ZTAsq6OqQ/Ia7Ne6DF/xJDOlfcS04WV/AGkSfxg"
    "YxAH3xbgs2Xnk1UXL4ljjfve4KJ9vneb8dm3qm42ztzmqcggy+Kd6oMlwpMnbEwU7bsMjglbk9OglBRv4hiOwTHasS8qUudW8IdU"
    "X6Dk5aEqjR6de81vXJqZ0XSbRACKM8AVO0PvAjcOHJb2vacX4K1/RnGTbeFnh+PRikVuViSrJ/0RLbQ/upt+bh2ti0+aa4ISMQAu"
    "d+Wl3pPY38mYB7MlwnyCw9p8avwJH5XvXhQVrqtq6w6z4dgAgyHLGobXXQfI/9TVboCMVpgnAmRiFc6p4zgEulNaD+/MEMrPbocX"
    "O3kBSeEMcaBADhW4E8/zRMZNOw9fZB7OoePUuhrN2AOlGLUP9Pqlpg7BcOi9ksfJKN19ykMfTVKPkyt1furHYLMJnBV++5yZNHYG"
    "fM0UX+8BAhtetLybuul3OWgX2v7zFCyBjM9Z9Up3dkobB+APcJZGVWuXDY+P9GOwLYrXYFROUS58xrsxtlR29BvV8mm/u2UUJN8R"
    "e3PA4/SoNOQ3rDKs6iX3NUFVPKXd3U8pmNqxzrhRew4t9qY6nv5bAxx1Ao+4EI7l73y5P1E34UtA/qCfbpvc5UEmuuExScDuM9xF"
    "NgmgT6ljsttFUMcx6AGZA8FtEMVwaTjGmX9x+0B/sUnntI2z6tIFOz9UvoZKdkfJIwcx7ddyEaRK6BhZ71WeOiB7NZ+HMZwlxAnp"
    "kepZH85xc4fDN6gU1rYIFy4OIb8lMvJs03Ulq5EJOzZmwcdzeKJyOgcrcLtvWajTpjiWT/pPGm9Lmt1HY+uyPMSFb0gX1mmM0Op/"
    "I41ZNBoCcHEmN5ye/d4sKnYNduUIN44/H+b4Ng4tH0Ra/3BmsFLyOOp5JIddfCusxUBLk5R3vBuccxdwf7INVOWK2Nvb9eDcQ4Hd"
    "rGKmGiLnD/eQPSyKGhr7X0U9GlnmeZdfyXiMq1f0+ddErub5NbGP+3j/RPqcnOriPJFoj1s+9jjWpfx3SdN/pw5/6/BsOguNw8l8"
    "cUU7/fmx4zQT0se1i4v6aEJoqdwwquyBiXev7lisGzX31uiosZF8BZhRVp0xZbJQmXNSQZ+FerBpPHi9MC6HwJJltKIX7Km2YTP7"
    "63TLYceATzXBxwR3lcQI4aPgInnXq3zpfNika0tsg+VRvY57JDEuaBzOJ6Z1378VZdz3u7tcgWWZbA7iXTgES3Sn73awiO6+1LWr"
    "pj3SSBioT610VKrzBfbGGWZ2ICq/pqFWmFvVnXkrNRonP4xwS5f5IpIle4qLlZhJL1cTO9hk9lRaisi0+EJHS13tqqUA1+USIXXz"
    "3krDe87mKReZ7U1HrXGx2h0zCOOuTwIfPc3JqBJxxvmwEr1byHn7+h7emqSz+iC2MSufeEvWPFpts3KUxra+00DRD9JkcNTg2Dlr"
    "RGloCEhr565ldWA8KIeV7AEVOne4rCrjyxXZ6yUopWHqI5dQhQwmpqu1qb6qC6pNxzdm4hneb3OGngpziPu/OUtJAG4FjetG7CN1"
    "Kv7rqktRe/XyzPBx4M2qh88EzwQpVGC5PoLatCUrD3p7QnwBmjGF5vwXY849b/SlkKwGGO+Z6stVkHskNtDS6tHFpMIavpwAuRay"
    "MUTa8gFueXNmElnE5PaHC9WU3pEpa9eR9LyOsi+QRC0DdU8cw+e4McfS7/S8OpmYZBkd6YivRGzcfRWwfFEQPGmDhst9HLVi6DD1"
    "Mp3gXuee1b0gufU9KlpXvdsdGb96fJP7fG4SS3qRzm+sB36/txzcwufaxvkeDtByqUQ1QvPuCJ05RgxBFXrEIiUnhOZeJzLQAtLk"
    "W1kJbtTebAvV5DDrwW6GzFI5OqX7VyivH8NNgP8HNmYzBmbq/Yj3uJ2U8LlQmKNxx0iXCEqdQMFBF+rCEFdJkPirbZAvOkCH40Rq"
    "DXFaFIAQwMhXoerzXK1cJrO+8oGOOnIxW3d60oJOpQOLB3LkpveHCoL8iExz1xAbRUi/nsfBljNuI30u0G4LY4mxgSE9ukQzFHew"
    "VF1JlxO0u9qNjnDCxua28UKYo357R4YFJF81a0g9qnMb5EVVScJ2R3i18jT6DLXxAQx7ljnjsXhMuxbjtgEZB8cBWSxUTshtnS8n"
    "AY15jGtd6vg1gfyWk9rxjKB/loJbKsX1WPOTw5RhfX71UeDy4TK4h+cIsegjrrWuoZhhG/rVUhn6xtXOVbizDuQ2IFxFZYvFvmdl"
    "7WBoOyDaiIYa8CbTCN84cQzamPajU65N+pumJpN9sSYjpw+aZkXzYpglejfuRp7fTBRQprhYRU9Q+Lng36k1GHWJyl+u3r1++frH"
    "Eb5RAqz2tObLfHE+9ptpnLv5Vl9tKfcvqQzcdRAvjXZ8hN3t8Ov9cd6NJIgDfn3SPkIhaTfHU27q3ArsjG8gqtARNL/mSW0GMGHX"
    "tD6r3cout3N8YwfJtSoY0wXNdClwpI0394yYg0JBUn3iWTC/4QwgxyZjjNfpA/ORqtRgq4hydVepUFInAdHs1v58VvVK81ImMfFu"
    "ArhPLB2TqiW6y+E6NnIYA9URGvXOcF3pGVeHSNC/dM66req6kneau8G944OEKpsDb9R1vAHf0VNVORPTGZUUwdia8iqiK/vNm1+q"
    "U1+o1LwIqqZKFH4xYcqdgXWSh2tKq1ELIQxbv0EwQWKFQR5HYW6pW9YNWjENJHBHTM9tpgmFtz1rKYcRHt1Kk5Ed7iUOXVY7d2ik"
    "4bSxr8sdGbI5S5UFJljfOGimHJdNsOMgirhEGIZKUtYQRAejI2AAMh4BQYXXf2A1N7C1JTOH8GYs/hdXHLaczcAg4wrifhlrP0wn"
    "ROALHLetQM6DGitbkN5FyGoYD6pvpCBLEH+Zl0ND5ycYEhWyje2rRbD5pdudgu+G4Hb0Aa3ifAn/PYIYhR/nyk16F+SbbearRQw1"
    "ivsLMrh244vw9Lz6QgnuTGVxTzuHbHzbBFc332ASnME95qhJldkts5IkbfnuDU4QH3NuRZUs3TngYVJs79AgphUjRVV/9Qe+PaP2"
    "zfSshhtnjBy07pEgKcb9LmeW9lDzDmv25MrVYGNF8g13NLWIU+2p7hqN+XUW5lG6GCMz7cmTmzv3xDTLNLHAJkM3jUKCyLYMCx8y"
    "Wn9tA9eUgL3TvicJMb9Ldc1/0442PVectcjfVVF/ydWhMdbj73hSUoNgXx2V4cwAyaQRBv8YI9eufT8r+WgUJyqZNwKWm0K7bP3L"
    "8758McrRUrS2Ne93upX6DveUeVCrs1FPzfE4s1blh8Y996Nlvdw8581vsiYvVBU6XAPpNp8rt4CxG+a3cRnG9GKbQ9Roj0CXL6A9"
    "b3O5DJ4WX+t2h1WLa3tspv+Sg6wCStOP9yVuscOfIy62xxxox51naluPTvFfQIj2lhLURW+s/TyGG6Hh8nGO6ulE7q5tRpn4GCBP"
    "WqJa1UVqMKLVXax18UlzU+tt7PSX1l6Rqv7nMoWrioKS95kmBclrcpAjEJRjmwm4Z6kvjlNMtCFAYW9oFl9Em4xUjp2qp3i3EZwk"
    "LqCvVgtmBf46NCBfQKn3CCPWMbEjTIEa1eypyaA5LVF86YCamW9pDqFRSOc3I+Sl8N+tsmIPkAfb9gVKhegRz9RtgmNbbrA3ISVy"
    "SB4+Q1W3SF4sKjH6iEB0O6NdOnRDevVFv0oeU6fKvFnKh6ovkLNxbkZ6+KX/zcWxFlK/M56ku+AezLvI6pvl5u60q1+0mERT7vv3"
    "MZvum7B1rzIFH0Gl+9ZVZ9Wss9KXMI7cLlPFcyqWI6h1tI5gVWfXpqqDi984TadEohkg4vti2PNFK9sVe6wb+XDUpPu75fZdqgPe"
    "fMoMmc1fXFiJZRgJcVl4n6LMru/WVF/75/13lP2ABX3K+NKjXl3w8q3//fUPr64+4PxcUFifuq4rNOJ7PeuxcHzvQMloORc+SXwQ"
    "npr2lUl8r746Aa5XMy/4MCGEgMTe+byUlKnr6xHn3XJIlThbHt3z+9aoWbd6tEwOL6ToDqq1pr5kM/pMwf7sYZnsG/dqG4FPeDsU"
    "DzNPkTmmwtImT7uLXjWiCg88Rkb6UJoMuQd6/5oAkWCufuKpO9U1mPiSWuTAf8qgsQ7CS7ng86fv3Oro5oaPITVTiAKAU3+npXeV"
    "rxin3+JJX5dWbOEGCTK2CumBq+YFaTcF8c35Roznf26jnMwR414eOCy3M24lTRwbX0qigcsFgRrQsflrQI/0dFBVaSP4rjTkcvP9"
    "9+pW1vGzY404qavRBIfHx/Yi3zFCjeQLa9l5x7dpqED6sf50VL1rEtWFl0cng2vdTtXxn8cr6etPHq1026540wX+oMK2m4N+FkU3"
    "9G++fKMOq/LlNKebWSeQvtFfn7btmKvcO6Bmuz3sOJ93z2H75dMtO4ZV99dK+RfD6LAm8ldaVWH3paQXFGOVtIO8OPdYB8dnfVAV"
    "1qoGMEtqjdmcIvOt+ppQcUyHuB0mZny3nGfu0ZWK06Nz2wa6eyYXvuRIfZmv8k2totsQKUiKro4PYn6T79FK4tqpJ3Ks3mM84fxo"
    "58mpMnY629Xn1Y9iBKt6dt2Es37qrXbUsyh/yhYQdsooxzaRYrQPwh9H1fc34RA/iGBU3QJKbxRhjIyL3Xr1rbZVsuh+Enj0MHUC"
    "sP8I2fVKkUEOje9DGPi+yqARyXDy/wBnxcVy"
)
SCRIPT = os.path.join(BASE, "m8_train_full.py")
open(SCRIPT, "wb").write(zlib.decompress(base64.b64decode(EMBED)))
print("script:", SCRIPT, hashlib.sha256(open(SCRIPT, "rb").read()).hexdigest()[:12], "(expected 9df57d0f4d66)")
def run(args):
    p = subprocess.Popen([sys.executable, "-u", SCRIPT, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, "failed; read the lines above"

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.163", "albumentations==2.0.8", "onnx", "onnxruntime", "onnxslim"], check=True)
import torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
w = glob.glob("/kaggle/input/**/UVH-26-MV-YOLOv11-S.pt", recursive=True)
IISC = w[0] if w else None
if IISC is None and RUN == "s_iisc":
    for d in sorted(glob.glob("/kaggle/input/*/*/*")[:20] + glob.glob("/kaggle/input/*")[:20]):
        print("  attached:", d)
assert IISC or RUN != "s_iisc", "IISc's UVH-26-MV-YOLOv11-S.pt not found: add the dataset that contains it (uvh26_train_kaggle/bundle/models/UVH-26-MV-YOLOv11-S.pt, 55 MB) before the 40-minute data unpack"
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
def show_inputs():
    for d in sorted(glob.glob("/kaggle/input/*")):
        print("  ", d, "->", sorted(os.listdir(d))[:6])
sj = glob.glob("/kaggle/input/**/shards.json", recursive=True)
if sj:                                   # packed dataset from step 17a: unpack onto the fast local disk
    DS = f"{TMP}/ds"
    t1 = time.time(); run(["unpack", "--src", os.path.dirname(sj[0]), "--out", DS])
    print(f"unpacked in {(time.time() - t1) / 60:.0f} min")
else:
    mh = glob.glob("/kaggle/input/**/dataset_manifest.json", recursive=True)
    if not mh:
        print("what is attached under /kaggle/input:"); show_inputs()
    assert mh, "step 17a data not found: Add Input -> your uvh26-train-1280 dataset (it holds shards.json and shard_*.tar), then run again"
    DS = os.path.dirname(mh[0])
    assert glob.glob(f"{DS}/images/train/*.jpg")[:1], "dataset has a manifest but no training images; rerun step 17a (older versions lost files to Kaggle's 500-file output limit)"
print("dataset:", DS)
OUT = f"{BASE}/run_{RUN}"

In [ ]:
t0 = time.time()
args = ["train", "--ds", DS, "--run", RUN, "--time", str(TIME_H), "--out", OUT, "--workers", "4"]
if IISC:
    args += ["--iisc-weights", IISC]
run(args)
print(f"finished in {(time.time() - t0) / 3600:.1f} h")
shutil.copy(f"{OUT}/uvh26_{RUN}_results.zip", f"{BASE}/uvh26_{RUN}_results.zip")